In [5]:
import nltk
import re
import math
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import ne_chunk, pos_tag
from nltk.data import find

# --- SETUP ---
def initialize_nltk():
    resources = ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'averaged_perceptron_tagger_eng', 'maxent_ne_chunker', 'maxent_ne_chunker_tab', 'words']
    for res in resources:
        nltk.download(res, quiet=True)
    return WordNetLemmatizer()

def check_nltk_resources():
    """Ensure all required NLTK resources are available"""
    required_resources = ['maxent_ne_chunker', 'maxent_ne_chunker_tab', 'words']
    for resource in required_resources:
        try:
            find(f'tokenizers/{resource}')
        except LookupError:
            nltk.download(resource, quiet=True)

def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

def clean_text(text, lemmatizer):
    tokens = word_tokenize(text)
    cleaned = []
    for t in tokens:
        t_clean = re.sub(r"[^\w\s]", "", t).lower().strip()
        if not t_clean or re.search(r'\d', t_clean):
            continue
        lemma = lemmatizer.lemmatize(t_clean, get_wordnet_pos(t_clean))
        cleaned.append(lemma)

    print("\nTokenization:")
    print(tokens)

    return cleaned

def extract_named_entities(text):
    """Returns a list of named entities found in the text"""
    check_nltk_resources()  # Ensure resources are available
    sentences = sent_tokenize(text)
    named_entities = []
    for sentence in sentences:
        chunks = ne_chunk(pos_tag(word_tokenize(sentence)))
        for chunk in chunks:
            if hasattr(chunk, 'label'):
                entity = " ".join(c[0] for c in chunk)
                named_entities.append((entity, chunk.label()))
    
    print("\nNamed Entity Recognition:")
    print(named_entities)

    return named_entities

# --- MATH ---
def calculate_tfidf(all_docs):
    vocab = sorted(set([t for doc in all_docs for t in doc]))
    N = len(all_docs)
    tfs = []
    for doc in all_docs:
        tfs.append({w: doc.count(w) for w in vocab if w in doc})

    idf = {w: math.log((N + 1) / (sum(1 for doc in all_docs if w in doc) + 1)) + 1 for w in vocab}

    vectors = []
    for counts in tfs:
        vectors.append({w: counts.get(w, 0) * idf[w] for w in vocab})
    return vectors

def get_similarity(vec_a, vec_b):
    dot = sum(vec_a.get(k, 0) * vec_b.get(k, 0) for k in vec_a if k in vec_b)
    norm_a = math.sqrt(sum(v**2 for v in vec_a.values()))
    norm_b = math.sqrt(sum(v**2 for v in vec_b.values()))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

# --- MAIN ---
if __name__ == "__main__":
    lemmatizer = initialize_nltk()

    # Load requirements
    with open('requirements-zoom.txt', 'r', encoding='utf-8') as file:
        lines = [line.strip() for line in file if line.strip()]

    nfr_raw = lines[:3]
    # Use simple list for FRs to maintain FR1, FR2 numbering
    fr_raw = lines[3:]

    # Clean
    nfr_docs = [clean_text(t, lemmatizer) for t in nfr_raw]
    fr_docs = [clean_text(t, lemmatizer) for t in fr_raw]

    # Example: Perform Named Entity Recognition on the first NFR
    if nfr_raw:
        print("Named Entities in the first NFR:")
        try:
            print(extract_named_entities(nfr_raw[0]))
        except LookupError as e:
            print("Error performing NER:", e)

    # Vectorize
    all_vecs = calculate_tfidf(nfr_docs + fr_docs)
    nfr_vecs = all_vecs[:3]
    fr_vecs = all_vecs[3:]

    # THRESHOLDING LOGIC
    THRESHOLD = 0.17

    # print("Requirement,NFR1,NFR2,NFR3")
    with open("zoom_output.txt", "w") as f:
        for i, fr_v in enumerate(fr_vecs):
            # Calculate raw scores
            sims = [get_similarity(fr_v, n_v) for n_v in nfr_vecs]

            # Convert to Binary (1 if > threshold, else 0)
            binary_row = [1 if s >= THRESHOLD else 0 for s in sims]

            line = f"FR{i+1},{binary_row[0]},{binary_row[1]},{binary_row[2]}"
            # print(line)
            f.write(line + "\n")


Tokenization:
['NFR1', '(', 'Performance', ')', ':', 'The', 'system', 'shall', 'support', 'video', 'streaming', 'at', 'up', 'to', '60', 'frames', 'per', 'second', 'without', 'performance', 'degradation', '.']

Tokenization:
['NFR2', '(', 'Scalability', ')', ':', 'The', 'system', 'shall', 'support', 'multiple', 'simultaneous', 'high-definition', 'participant', 'streams', '.']

Tokenization:
['NFR3', '(', 'Quality', ')', ':', 'The', 'system', 'shall', 'deliver', 'high-resolution', 'video', 'output', 'up', 'to', '4K', 'display', 'capability', '.']

Tokenization:
['FR1', ':', 'The', 'system', 'shall', 'provide', '60fps', 'video', 'streaming', 'during', 'meetings', '.']

Tokenization:
['FR2', ':', 'The', 'system', 'shall', 'allow', 'users', 'to', 'enable', 'high-frame-rate', 'video', 'mode', '.']

Tokenization:
['FR3', ':', 'The', 'system', 'shall', 'support', 'high-bit-rate', 'video', 'streaming', '.']

Tokenization:
['FR4', ':', 'The', 'system', 'shall', 'adjust', 'bitrate', 'dynamically